# Baseline: Self-Attention (Transformer Encoder) for Unit Commitment Prediction

A second, deep-learning baseline for the same task as `baseline_random_forest.ipynb`, using a small
**non-causal (bidirectional) Transformer encoder** instead of a Random Forest.

Why non-causal rather than autoregressive: every exogenous input (day-ahead price, inflow) is known for
the *entire* 168-hour week in advance — there is no hidden future on the input side, so this is a
sequence-labeling problem, not a forecasting problem. The model attends over the whole known week at once
and predicts all 168x14 commitment decisions in a single parallel forward pass. An autoregressive decoder
that conditions each hour's decision on previous hours' *predicted* decisions (to capture minimum
up/down-time and start/stop-cost effects) is a plausible follow-up, deferred until this simpler model's
results motivate it.

The dataset is small for deep learning (2,922 labeled weekly cases, heavily overlapping daily rolling
windows), so the model and training setup below are deliberately small and regularized. The Random Forest
baseline's **0.62** validation micro-average ROC AUC is the number to beat.

A single training run's validation AUC turned out to be noisy here — plain weight-init randomness swings
it by several points (discovered while building `ablation_generator_attributes.ipynb`). So instead of one
run, this notebook: (1) trains the default config across 5 seeds and reports mean +/- std, (2) builds a
seed ensemble (averaged probabilities across those 5 seeds), (3) runs a small hyperparameter sweep scored
across multiple seeds, and (4) uses whichever config wins for the final submission, itself a 5-seed
ensemble.

(A separate ablation — conditioning the output head on each generator's static physical attributes — is
in `ablation_generator_attributes.ipynb`, kept out of this notebook so it stays focused. It did not
improve on the model here.)

## 1. Setup

In [ ]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score

from kaggle_metric import score as kaggle_score

HORIZON = 168  # hours in one weekly case

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

## 2. Load data

Same loading helpers as `baseline_random_forest.ipynb`.

In [ ]:
BASE_PATH = Path(".")
KERNEL_PATH = BASE_PATH / "data" / "kernel"
EXTENDED_PATH = BASE_PATH / "data" / "extended"


def load_series(path, date_col="date"):
    df = pd.read_csv(path, parse_dates=[date_col])
    df[date_col] = df[date_col].dt.tz_localize(None)
    return df


def series_arrays(df, date_col="date"):
    value_cols = [c for c in df.columns if c != date_col]
    return df[date_col].to_numpy(), df[value_cols].to_numpy(dtype=float), value_cols


price = load_series(KERNEL_PATH / "Historical_day_ahead_price_2015_2025.csv")
inflow = load_series(KERNEL_PATH / "Historical_inflow_1958_2025.csv")
volume = load_series(KERNEL_PATH / "Historical_volume_2015_2024.csv")
water_value = load_series(KERNEL_PATH / "Synthetic_water_value_2015_2024.csv")
min_flow = load_series(EXTENDED_PATH / "Constraint_min_flow.csv")
min_volume = load_series(EXTENDED_PATH / "Constraint_min_volume.csv")

uc = pd.read_csv(KERNEL_PATH / "Unit_commitment_decisions.csv", parse_dates=["starttime"])
prediction_mapping = pd.read_csv(BASE_PATH / "prediction_mapping.csv", parse_dates=["starttime"])
sample_submission = pd.read_csv(BASE_PATH / "sample_submission.csv")

label_cols = [c for c in uc.columns if c.startswith("result_committed")]

price_dates, price_values, _ = series_arrays(price)
inflow_dates, inflow_values, inflow_cols = series_arrays(inflow)
volume_dates, volume_values, volume_cols = series_arrays(volume)
water_value_dates, water_value_values, water_value_cols = series_arrays(water_value)
min_flow_dates, min_flow_values, min_flow_cols = series_arrays(min_flow)
min_volume_dates, min_volume_values, min_volume_cols = series_arrays(min_volume)

N_DYNAMIC = 1 + len(inflow_cols)
N_STATIC = len(volume_cols) + len(water_value_cols) + len(min_flow_cols) + len(min_volume_cols)
print(f"{len(label_cols)} label columns, {len(uc)} labeled weekly cases, "
      f"{len(prediction_mapping)} cases to predict for submission")
print(f"dynamic features per hour: {N_DYNAMIC} (price + {len(inflow_cols)} inflow columns)")
print(f"static context features: {N_STATIC}")

## 3. Build sequence tensors

Unlike the Random Forest baseline (one flat feature vector per week), each case becomes a
`(168, N_DYNAMIC)` sequence of per-hour price + inflow, plus a `(N_STATIC,)` static context vector
(start-of-week volume / water value / min-flow / min-volume — identical start-of-week lookups to the RF
notebook's `build_features`, just kept separate instead of concatenated into one flat row).

The 2,352 label columns are **generator-major** (confirmed from the file header: all 168 `_t0..t167`
columns for one generator are contiguous before the next generator's block starts), so the generator name
order is derived programmatically from the column names rather than hardcoded, and a round-trip assertion
checks the reshape is correct before any training happens.

In [ ]:
def build_sequence_features(starttimes):
    starttimes = pd.to_datetime(starttimes).to_numpy()

    price_pos = np.searchsorted(price_dates, starttimes)
    assert np.all(price_dates[price_pos] == starttimes), "starttime not aligned with price grid"
    idx168 = price_pos[:, None] + np.arange(HORIZON)[None, :]
    price_seq = price_values[idx168]  # (n, 168, 1)

    inflow_pos = np.searchsorted(inflow_dates, starttimes)
    assert np.all(inflow_dates[inflow_pos] == starttimes), "starttime not aligned with inflow grid"
    inflow_idx168 = inflow_pos[:, None] + np.arange(HORIZON)[None, :]
    inflow_seq = inflow_values[inflow_idx168]  # (n, 168, len(inflow_cols))

    dynamic_seq = np.concatenate([price_seq, inflow_seq], axis=-1).astype(np.float32)

    volume_feat = volume_values[np.searchsorted(volume_dates, starttimes)]
    water_value_feat = water_value_values[np.searchsorted(water_value_dates, starttimes)]
    min_flow_feat = min_flow_values[np.searchsorted(min_flow_dates, starttimes)]
    min_volume_feat = min_volume_values[np.searchsorted(min_volume_dates, starttimes)]
    static_ctx = np.hstack(
        [volume_feat, water_value_feat, min_flow_feat, min_volume_feat]
    ).astype(np.float32)

    return dynamic_seq, static_ctx


# Derive generator order from the label columns themselves (generator-major blocks of 168).
_pattern = re.compile(r"^result_committed_(.+)_t(\d+)$")
generator_names = []
_seen = set()
for c in label_cols:
    name = _pattern.match(c).group(1)
    if name not in _seen:
        _seen.add(name)
        generator_names.append(name)
n_generators = len(generator_names)
print(f"{n_generators} generators, in column order: {generator_names}")

uc_sorted = uc.sort_values("starttime").reset_index(drop=True)
dynamic_all, static_all = build_sequence_features(uc_sorted["starttime"])

Y_flat = uc_sorted[label_cols].to_numpy(dtype=np.float32)  # (N, 2352), generator-major
N = Y_flat.shape[0]
Y_all = Y_flat.reshape(N, n_generators, HORIZON).transpose(0, 2, 1)  # (N, 168, 14)

# Round-trip assertion: inverting the reshape must reproduce the original flat label layout exactly.
Y_roundtrip = Y_all.transpose(0, 2, 1).reshape(N, -1)
assert np.array_equal(Y_roundtrip, Y_flat), "label reshape does not round-trip correctly"

print("dynamic_all:", dynamic_all.shape, "static_all:", static_all.shape, "Y_all:", Y_all.shape)

pos_rates = pd.Series(Y_all.reshape(-1, n_generators).mean(axis=0), index=generator_names)
print("\nPer-generator commitment rate (fraction of hours ON):")
print(pos_rates.sort_values())

## 4. Time-based train / validation split

Identical logic to `baseline_random_forest.ipynb`: hold out the most recent ~15% of cases by
`starttime`, since cases are daily rolling 168h windows that overlap heavily and a random split would
leak near-duplicate windows across train/validation.

In [ ]:
n_val = int(0.15 * len(uc_sorted))

dynamic_train, dynamic_val = dynamic_all[:-n_val], dynamic_all[-n_val:]
static_train, static_val = static_all[:-n_val], static_all[-n_val:]
Y_train, Y_val = Y_all[:-n_val], Y_all[-n_val:]
run_no_train = uc_sorted["Run No"].iloc[:-n_val].to_numpy()
run_no_val = uc_sorted["Run No"].iloc[-n_val:].to_numpy()

print(f"train: {dynamic_train.shape[0]} cases ({uc_sorted['starttime'].iloc[0].date()} to "
      f"{uc_sorted['starttime'].iloc[-n_val - 1].date()})")
print(f"val:   {dynamic_val.shape[0]} cases ({uc_sorted['starttime'].iloc[-n_val].date()} to "
      f"{uc_sorted['starttime'].iloc[-1].date()})")

## 5. Normalize

Mean/std are fit on the **training split only** (to avoid leakage) and reused as-is for validation and,
later, for the unlabeled `prediction_mapping` cases.

In [ ]:
dyn_mean = dynamic_train.reshape(-1, dynamic_train.shape[-1]).mean(axis=0)
dyn_std = dynamic_train.reshape(-1, dynamic_train.shape[-1]).std(axis=0) + 1e-6
static_mean = static_train.mean(axis=0)
static_std = static_train.std(axis=0) + 1e-6


def normalize(dynamic_seq, static_ctx):
    return (dynamic_seq - dyn_mean) / dyn_std, (static_ctx - static_mean) / static_std


dynamic_train_n, static_train_n = normalize(dynamic_train, static_train)
dynamic_val_n, static_val_n = normalize(dynamic_val, static_val)

## 6. Dataset / DataLoader

In [ ]:
class UCDataset(torch.utils.data.Dataset):
    def __init__(self, dynamic_seq, static_ctx, target):
        self.dynamic_seq = torch.as_tensor(dynamic_seq, dtype=torch.float32)
        self.static_ctx = torch.as_tensor(static_ctx, dtype=torch.float32)
        self.target = torch.as_tensor(target, dtype=torch.float32)

    def __len__(self):
        return len(self.dynamic_seq)

    def __getitem__(self, idx):
        return self.dynamic_seq[idx], self.static_ctx[idx], self.target[idx]


train_dataset = UCDataset(dynamic_train_n, static_train_n, Y_train)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)

dynamic_val_t = torch.as_tensor(dynamic_val_n, dtype=torch.float32).to(device)
static_val_t = torch.as_tensor(static_val_n, dtype=torch.float32).to(device)

print(f"{len(train_dataset)} training sequences, batch size 32 -> {len(train_loader)} batches/epoch")

## 7. Model

A small Transformer encoder: project each hour's (price, inflow) into `d_model`, prepend one extra
token projected from the static start-of-week context, add a learned positional embedding, run through
a couple of non-causal (bidirectional) self-attention layers, then apply one shared `Linear(d_model, 14)`
head per hour to produce the 14 generator logits for that hour.

Kept deliberately small (`d_model=64`, 2 layers, 4 heads) given ~2,484 overlapping training sequences —
more capacity than this risks overfitting long before it helps.

In [ ]:
class UCTransformer(nn.Module):
    def __init__(self, n_dynamic=N_DYNAMIC, n_static=N_STATIC, n_generators=n_generators,
                 horizon=HORIZON, d_model=64, n_heads=4, n_layers=2, d_ff=128, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(n_dynamic, d_model)
        self.ctx_proj = nn.Linear(n_static, d_model)
        self.pos_embedding = nn.Parameter(torch.randn(horizon + 1, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.output_head = nn.Linear(d_model, n_generators)

    def forward(self, dynamic_seq, static_ctx):
        hourly_tokens = self.input_proj(dynamic_seq)          # (batch, 168, d_model)
        ctx_token = self.ctx_proj(static_ctx).unsqueeze(1)    # (batch, 1, d_model)
        tokens = torch.cat([ctx_token, hourly_tokens], dim=1)  # (batch, 169, d_model), no causal mask
        tokens = tokens + self.pos_embedding.unsqueeze(0)
        encoded = self.encoder(tokens)                        # (batch, 169, d_model)
        hourly_encoded = encoded[:, 1:, :]                    # drop the context-token position
        return self.output_head(hourly_encoded)                # (batch, 168, 14) logits


n_params = sum(p.numel() for p in UCTransformer().parameters())
print(f"UCTransformer parameter count: {n_params:,}")

## 8. Training infrastructure

`BCEWithLogitsLoss` over all 168x14 outputs at once, `AdamW` with weight decay, early stopping on
validation micro-average ROC AUC (via the actual `kaggle_metric.score`), and a reseed right before each
model's construction so every run gets a clean, isolated init+shuffle stream regardless of what ran
before it in the notebook (see `ablation_generator_attributes.ipynb` for why that matters: without it,
"the same" model can land on measurably different scores just from incidental earlier RNG draws).

`run_seeded_training` wraps one full seeded training run (model construction -> early-stopped training
-> best checkpoint's validation probabilities), reused below for the multi-seed evaluation, the
hyperparameter sweep, and the final winning-config run.

In [ ]:
def predictions_to_submission_frame(proba, run_numbers):
    # proba: (n_cases, 168, 14) -> flatten back to the generator-major column layout of label_cols
    proba_flat = proba.transpose(0, 2, 1).reshape(proba.shape[0], -1)
    df = pd.DataFrame(proba_flat, columns=label_cols)
    df.insert(0, "Run No", run_numbers)
    return df


Y_val_flat = Y_val.transpose(0, 2, 1).reshape(Y_val.shape[0], -1)
solution_df = pd.DataFrame(Y_val_flat, columns=label_cols)
solution_df.insert(0, "Run No", run_no_val)

loss_fn = nn.BCEWithLogitsLoss()


def compute_val_auc(model):
    model.eval()
    with torch.no_grad():
        proba = torch.sigmoid(model(dynamic_val_t, static_val_t)).cpu().numpy()
    return kaggle_score(solution_df, predictions_to_submission_frame(proba, run_no_val))


dynamic_train_t = torch.as_tensor(dynamic_train_n, dtype=torch.float32).to(device)
static_train_t = torch.as_tensor(static_train_n, dtype=torch.float32).to(device)
Y_train_flat = Y_train.transpose(0, 2, 1).reshape(Y_train.shape[0], -1)
solution_train_df = pd.DataFrame(Y_train_flat, columns=label_cols)
solution_train_df.insert(0, "Run No", run_no_train)


def compute_train_auc(model):
    # Scored the same way as compute_val_auc, but against the training split -- the gap between the
    # two (if any) is the overfitting signal: training AUC climbing while validation AUC plateaus/falls.
    model.eval()
    with torch.no_grad():
        proba = torch.sigmoid(model(dynamic_train_t, static_train_t)).cpu().numpy()
    return kaggle_score(solution_train_df, predictions_to_submission_frame(proba, run_no_train))


def run_seeded_training(seed, model_kwargs, lr=3e-4, weight_decay=1e-2,
                         max_epochs=150, patience=15, log_prefix="", track_history=False):
    torch.manual_seed(seed)
    model = UCTransformer(**model_kwargs).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_auc, best_epoch, best_state = -1.0, -1, None
    epochs_without_improvement = 0
    history = [] if track_history else None
    start = time.time()
    for epoch in range(max_epochs):
        model.train()
        for dyn, stat, tgt in train_loader:
            dyn, stat, tgt = dyn.to(device), stat.to(device), tgt.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(dyn, stat), tgt)
            loss.backward()
            optimizer.step()

        val_auc = compute_val_auc(model)
        if track_history:
            history.append((epoch, compute_train_auc(model), val_auc))
        if val_auc > best_auc:
            best_auc, best_epoch = val_auc, epoch
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        best_val_proba = torch.sigmoid(model(dynamic_val_t, static_val_t)).cpu().numpy()
    print(f"{log_prefix}seed {seed}: best_auc {best_auc:.4f} (epoch {best_epoch}, {time.time() - start:.1f}s)")
    return best_auc, best_epoch, best_val_proba, history


DEFAULT_CFG = dict(
    model_kwargs=dict(d_model=64, n_heads=4, n_layers=2, d_ff=128, dropout=0.2),
    lr=3e-4, weight_decay=1e-2,
)

## 9. Multi-seed evaluation (default config)

A single run's validation AUC is noisy: `ablation_generator_attributes.ipynb` found weight-init alone
swings this model's score by ~4 AUC points on this dataset. Training the default config across several
seeds gives a mean +/- std instead of one potentially-lucky-or-unlucky number, and is also the input to
the seed ensemble below.

In [ ]:
SEEDS = [0, 1, 2, 3, 4]

default_results = [
    run_seeded_training(seed, **DEFAULT_CFG, log_prefix="[default] ") for seed in SEEDS
]
default_aucs = np.array([r[0] for r in default_results])
print(f"\ndefault config across {len(SEEDS)} seeds: mean {default_aucs.mean():.4f} "
      f"+/- {default_aucs.std():.4f} (min {default_aucs.min():.4f}, max {default_aucs.max():.4f}, "
      f"RF baseline was ~0.62)")

## 10. Seed ensemble (default config)

Average the predicted probabilities across the 5 seeded models' best checkpoints and score that
ensemble the same way. Ensembling over random seeds is close to free (the models are already trained)
and should both reduce variance and usually beat any single seed.

In [ ]:
ensemble_proba = np.mean([r[2] for r in default_results], axis=0)
ensemble_auc = kaggle_score(solution_df, predictions_to_submission_frame(ensemble_proba, run_no_val))

print(f"default config seed-ensemble val AUC: {ensemble_auc:.4f}")
print(f"compare: single-seed mean {default_aucs.mean():.4f} +/- {default_aucs.std():.4f}, "
      f"best single seed {default_aucs.max():.4f}")

## 11. Hyperparameter sweep

A small, deliberately short grid (one more-regularized variant, one smaller model, one larger model),
each scored across 3 seeds (not 5, to keep runtime bounded) via mean validation AUC -- the sweep reuses
the default config's first 3 seeds from Section 9 instead of retraining them.

In [ ]:
SWEEP_SEEDS = [0, 1, 2]

variant_configs = {
    "default": DEFAULT_CFG,
    "more_regularized": dict(
        model_kwargs=dict(d_model=64, n_heads=4, n_layers=2, d_ff=128, dropout=0.3),
        lr=3e-4, weight_decay=5e-2,
    ),
    "smaller": dict(
        model_kwargs=dict(d_model=32, n_heads=4, n_layers=1, d_ff=64, dropout=0.2),
        lr=3e-4, weight_decay=1e-2,
    ),
    "larger": dict(
        model_kwargs=dict(d_model=128, n_heads=4, n_layers=3, d_ff=256, dropout=0.2),
        lr=3e-4, weight_decay=1e-2,
    ),
}

sweep_aucs = {"default": default_aucs[:len(SWEEP_SEEDS)]}
for name, cfg in variant_configs.items():
    if name == "default":
        continue
    sweep_aucs[name] = np.array([
        run_seeded_training(seed, **cfg, log_prefix=f"[{name}] ")[0] for seed in SWEEP_SEEDS
    ])

sweep_summary = pd.DataFrame(
    {name: {"mean_auc": aucs.mean(), "std_auc": aucs.std()} for name, aucs in sweep_aucs.items()}
).T.sort_values("mean_auc", ascending=False)
sweep_summary

## 12. Winning config: full 5-seed evaluation + ensemble

If the sweep's winner is "default", reuse Section 9/10's results directly instead of retraining
(they're identical runs). Otherwise, run the winning config across the full 5 seeds for a fair
comparison to Section 10's default ensemble.

In [ ]:
winner_name = sweep_summary.index[0]
winner_cfg = variant_configs[winner_name]
print(f"sweep winner: {winner_name} {winner_cfg}")

if winner_name == "default":
    winner_results, winner_aucs = default_results, default_aucs
    winner_ensemble_proba, winner_ensemble_auc = ensemble_proba, ensemble_auc
else:
    winner_results = [
        run_seeded_training(seed, **winner_cfg, log_prefix=f"[winner:{winner_name}] ") for seed in SEEDS
    ]
    winner_aucs = np.array([r[0] for r in winner_results])
    winner_ensemble_proba = np.mean([r[2] for r in winner_results], axis=0)
    winner_ensemble_auc = kaggle_score(
        solution_df, predictions_to_submission_frame(winner_ensemble_proba, run_no_val)
    )

print(f"\nwinner ({winner_name}) single-seed: mean {winner_aucs.mean():.4f} +/- {winner_aucs.std():.4f}")
print(f"winner ({winner_name}) seed-ensemble: {winner_ensemble_auc:.4f}")
print(f"(RF baseline was ~0.62)")

## 13. Per-generator breakdown

Computed on the winning config's seed-ensemble predictions (the best overall estimate built above).

In [ ]:
per_gen_auc = {}
for i, name in enumerate(generator_names):
    y_true = Y_val[:, :, i].reshape(-1)
    y_score = winner_ensemble_proba[:, :, i].reshape(-1)
    per_gen_auc[name] = roc_auc_score(y_true, y_score) if len(np.unique(y_true)) > 1 else float("nan")

pd.Series(per_gen_auc, name="val_auc").sort_values()

## 14. Build a submission for the unlabeled prediction cases

Refit the winning config across all 5 seeds on *all* labeled cases (fixed epoch budget: the median
best-epoch across the winner's 5 validation runs, +1 -- rather than re-running early stopping with no
held-out set left), then average the 5 refit models' predicted probabilities for the submission, mirroring
the seed-ensemble approach used for validation.

In [ ]:
winner_best_epochs = [r[1] for r in winner_results]
final_epochs = int(np.median(winner_best_epochs)) + 1
print(f"refitting {len(SEEDS)} seeds on all {len(uc_sorted)} labeled cases "
      f"for {final_epochs} epochs each (median best epoch + 1)")

dynamic_all_n, static_all_n = normalize(dynamic_all, static_all)
full_dataset = UCDataset(dynamic_all_n, static_all_n, Y_all)
full_loader = torch.utils.data.DataLoader(full_dataset, batch_size=32, shuffle=True)

dynamic_submit, static_submit = build_sequence_features(prediction_mapping["starttime"])
dynamic_submit_n, static_submit_n = normalize(dynamic_submit, static_submit)
dyn_submit_t = torch.as_tensor(dynamic_submit_n, dtype=torch.float32).to(device)
stat_submit_t = torch.as_tensor(static_submit_n, dtype=torch.float32).to(device)

submit_probas = []
start = time.time()
for seed in SEEDS:
    torch.manual_seed(seed)
    final_model = UCTransformer(**winner_cfg["model_kwargs"]).to(device)
    final_optimizer = torch.optim.AdamW(
        final_model.parameters(), lr=winner_cfg["lr"], weight_decay=winner_cfg["weight_decay"]
    )
    for epoch in range(final_epochs):
        final_model.train()
        for dyn, stat, tgt in full_loader:
            dyn, stat, tgt = dyn.to(device), stat.to(device), tgt.to(device)
            final_optimizer.zero_grad()
            loss = loss_fn(final_model(dyn, stat), tgt)
            loss.backward()
            final_optimizer.step()
    final_model.eval()
    with torch.no_grad():
        submit_probas.append(torch.sigmoid(final_model(dyn_submit_t, stat_submit_t)).cpu().numpy())
print(f"refit {len(SEEDS)} seeds in {time.time() - start:.1f}s")

submit_proba_ensemble = np.mean(submit_probas, axis=0)
submission = predictions_to_submission_frame(submit_proba_ensemble, prediction_mapping["Run No"].to_numpy())
submission = submission[sample_submission.columns]  # match exact column order

assert submission.shape == sample_submission.shape
assert list(submission.columns) == list(sample_submission.columns)

submission.to_csv("transformer_submission.csv", index=False)
submission.head()

## 15. Overfitting diagnostic: train vs. validation learning curves

Early stopping already protects the reported validation AUC from the worst of it, but it doesn't make
the train/validation gap *visible*. This re-trains the winning config once more (fixed seed, purely for
this diagnostic -- not used for any score reported above) tracking both training-set and validation-set
AUC every epoch, scored identically via `kaggle_metric.score` just on different splits.

Classic overfitting signature: training AUC keeps climbing toward 1.0 well past the point where
validation AUC plateaus or drops. If the two curves stay close together near the best epoch, that's
reassuring; if they diverge sharply, it's a sign to increase regularization (dropout/weight decay) or
lower `patience` rather than trust the gap away.

In [ ]:
import matplotlib.pyplot as plt

_, _, _, diagnostic_history = run_seeded_training(
    seed=0, **winner_cfg, log_prefix="[diagnostic] ", track_history=True
)
diagnostic_df = pd.DataFrame(diagnostic_history, columns=["epoch", "train_auc", "val_auc"])
diagnostic_best_epoch = diagnostic_df.loc[diagnostic_df["val_auc"].idxmax(), "epoch"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(diagnostic_df["epoch"], diagnostic_df["train_auc"], label="train AUC")
ax.plot(diagnostic_df["epoch"], diagnostic_df["val_auc"], label="val AUC")
ax.axvline(diagnostic_best_epoch, color="gray", linestyle="--", label=f"best epoch ({diagnostic_best_epoch})")
ax.set_xlabel("epoch")
ax.set_ylabel("micro-average ROC AUC")
ax.set_title(f"Train vs. validation AUC ({winner_name} config, seed 0)")
ax.legend()
plt.show()

gap_at_best = (
    diagnostic_df.loc[diagnostic_df["epoch"] == diagnostic_best_epoch, "train_auc"].iloc[0]
    - diagnostic_df.loc[diagnostic_df["epoch"] == diagnostic_best_epoch, "val_auc"].iloc[0]
)
print(f"train-val AUC gap at the best epoch: {gap_at_best:.4f}")
print(f"final train-val AUC gap (last epoch trained): "
      f"{diagnostic_df['train_auc'].iloc[-1] - diagnostic_df['val_auc'].iloc[-1]:.4f}")